In [ ]:
# import libraries
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load dataset
df=pd.read_csv("weatherAUS.csv")
df.head()
df.shape

In [ ]:
# handling date
df['Date']=pd.to_datetime(df['Date'])
df['year']=df['Date'].dt.year
df['month']=df['Date'].dt.month
df['day_of_week']=df['Date'].dt.dayofweek

# handle month and day with cyclic encoding

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df.drop(["month", "day_of_week"], axis=1, inplace=True)
df.head()

In [ ]:
# handle missing values
df.isnull().sum()
df.info()
# handle categorical columns missing values
for col in cat_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# handle numerical columns missing values
num_columns = df.select_dtypes(exclude='object').columns.tolist()

# fill missing values
df[cat_columns] = df[cat_columns].fillna(df[cat_columns].mode())
df[num_columns] = df[num_columns].fillna(df[num_columns].mean())
df.isnull().sum()

In [ ]:
# handle cat columns ()
# LabelEncoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

df['RainToday']=le.fit_transform(df['RainToday'])
df['RainTomorrow']=le.fit_transform(df['RainTomorrow'])

# one-hot encoding
df = pd.get_dummies(
    df,
    columns=['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm'],
    drop_first=True
)
# df.head()
df.shape


In [ ]:
# divide data into features and target
X=df.drop(['RainTomorrow', 'Date'],axis=1)
y=df['RainTomorrow']
X.head()
df.isnull().sum()

In [ ]:
# divide data into train test
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

# scaling data
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [ ]:
# import libraries for ANN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
#Early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    min_delta=0.001, # minimium amount of change to count as an improvement
    patience=20, # how many epochs to wait before stopping
    restore_best_weights=True,
)

# Initialising the model
model = Sequential()



In [ ]:
# define layers
model.add(Dense(64, activation='relu', input_shape=(115,)))
model.add(Dropout(0.25))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
# compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_history = model.fit(X_train, y_train, batch_size = 32, epochs = 100, callbacks=early_stopping, validation_split=0.33)

In [ ]:
# predict and evaluate test set results
y_pred=model.predict(X_test)
y_pred=(y_pred>=0.5)

In [ ]:
# visualize confusion matrix using direct confusion matrix not seaborn
from sklearn.metrics import ConfusionMatrixDisplay,classification_report

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, cmap="Blues"
)
plt.title("Confusion Matrix - Rain Prediction")
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

In [ ]:
# summarize history from model
plt.plot(model_history.history['accuracy'])
plt.plot(model_history.history['val_accuracy'])
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend(['train','test'],loc='best')
plt.title('model accuracy')
plt.show()

In [ ]:
# summarize history for loss
plt.plot(model_history.history['loss'])
plt.plot(model_history.history['val_loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('model loss')
plt.legend(['train','test'],loc='best')
plt.show()

In [ ]:
# calculate the accuracy
from sklearn.metrics import accuracy_score
score=accuracy_score(y_test,y_pred)
score